# PM2.5 Forecasting Baseline
Nachat Jatusripitak

In [29]:
# Import required packages for data management and evaluation
import pandas as pd
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [20]:
# Load dataset as CSV
df = pd.read_csv('../dataset_1_pixels_grid_indices.csv')
df['date'] = pd.to_datetime(df['date']) 
df = df.sort_values(['row', 'col', 'date'])

## Question 1: How hard is it to predict *level* of PM2.5?

I use a persistence baseline to evaluate the difficulty of predicting the level of PM2.5 from day-to-day.<br>In particular, I predict
$\hat PM2.5_t = PM2.5_{t-1}$ and compute MSE, RMSE, MAE, and R-squared.

In [25]:
# Create the lagged column
df = df.sort_values(['row', 'col', 'date'])
df['pm25_yesterday'] = df.groupby(['row', 'col'])['pm25_today'].shift(1)
df = df.dropna()


We find significant autocorrelation between PM2.5 values from day-to-day. A naive persistence model (essentially a random walk) achieves R-squared of 0.8 with a RMSE of 8.35.

This result indicates that predicting *changes* in PM2.5 is a much more interesting (and difficult) task.

In [26]:
today = df["pm25_today"]

yesterday = df["pm25_yesterday"]

mse = mean_squared_error(today, yesterday)
rmse = root_mean_squared_error(today, yesterday)
mae = mean_absolute_error(today, yesterday)
r2 = r2_score(today, yesterday)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R^2: {r2}")

MSE: 69.75688575692648
RMSE: 8.352058773555564
MAE: 3.8074202064629943
R^2: 0.803913501181029


# Question 2: How hard is it to predict $\Delta PM_{2.5}$?

In [27]:
# Create the lagged column
df = df.sort_values(['row', 'col', 'date'])
df['pm25_change_yesterday'] = df.groupby(['row', 'col'])['pm25_change'].shift(1)
df = df.dropna()

After differencing, we find that a naive persistence model performs poorly, as expected. We see that predicting $\Delta PM_{2.5}$ is a much harder task than predicting $PM_{2.5}$ levels alone.

In [31]:
today = df["pm25_change"]
yesterday = df["pm25_change_yesterday"]

mse = mean_squared_error(today, yesterday)
rmse = root_mean_squared_error(today, yesterday)
mae = mean_absolute_error(today, yesterday)
r2 = r2_score(today, yesterday)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R^2: {r2}")

zeros = np.zeros_like(today)

mse = mean_squared_error(today, zeros)
rmse = root_mean_squared_error(today, zeros)
mae = mean_absolute_error(today, zeros)
r2 = r2_score(today, zeros)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R^2: {r2}")

MSE: 151.17694557168278
RMSE: 12.295403432652495
MAE: 5.6982255613426505
R^2: -1.2670367443906403
MSE: 66.68665157806709
RMSE: 8.166189538460829
MAE: 3.778476365028878
R^2: -2.7411032520463507e-05
